# Fase 3: Estudio Comparativo Detallado y Análisis de Errores
En este cuaderno evaluaremos el rendimiento predictivo de diferentes algoritmos de Machine Learning de manera individual y progresiva. 

Para cada algoritmo, analizaremos su comportamiento frente a los cuatro escenarios de características (Crudo, Manual, LASSO, PCA), apoyándonos en la **Matriz de Confusión** para evaluar clínicamente los Falsos Positivos y, críticamente, los **Falsos Negativos** (tumores malignos no detectados).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings('ignore')

# Cargar la variable objetivo y los escenarios
y = pd.read_csv('./data/y_target.csv')['Diagnosis']
X_escenarios = {
    '1. Crudo (30 var)': pd.read_csv('./data/X_escenario_1_crudo.csv'),
    '2. Manual (14 var)': pd.read_csv('./data/X_escenario_2_manual.csv'),
    '3. LASSO (15 var)': pd.read_csv('./data/X_escenario_3_lasso.csv'),
    '4. PCA (10 comp)': pd.read_csv('./data/X_escenario_4_pca.csv')
}
print("Datos cargados correctamente.")

Datos cargados correctamente.


## 1. Función de Evaluación y Análisis Visual
Para mantener el código limpio y enfocado en el análisis, creamos una función que toma un algoritmo, realiza el `GridSearchCV` en los 4 escenarios de datos y grafica sus Matrices de Confusión de forma comparativa.

In [6]:
def analizar_modelo_en_escenarios(nombre_modelo, modelo_base, parametros):
    print(f"{"="*50}")
    print(f"🚀 INICIANDO ANÁLISIS: {nombre_modelo.upper()}")
    
    # Configurar el lienzo para las 4 matrices de confusión (1 fila, 4 columnas)
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(f'Matrices de Confusión - {nombre_modelo}', fontsize=16, fontweight='bold', y=1.05)
    
    resultados_modelo = []
    
    for idx, (nombre_escenario, X_actual) in enumerate(X_escenarios.items()):
        # División de datos
        X_train, X_test, y_train, y_test = train_test_split(X_actual, y, test_size=0.2, random_state=30, stratify=y)
        
        # GridSearch
        grid = GridSearchCV(modelo_base, parametros, cv=5, scoring='recall', n_jobs=-1)
        grid.fit(X_train, y_train)
        mejor_modelo = grid.best_estimator_
        
        # Predicción
        y_pred = mejor_modelo.predict(X_test)
        
        # Métricas
        recall = recall_score(y_test, y_pred) * 100
        accuracy = accuracy_score(y_test, y_pred) * 100
        
        print(f"📊 {nombre_escenario}")
        print(f"   Mejores Hiperparámetros: {grid.best_params_}")
        print(f"   Accuracy: {accuracy:.2f}%  |  Recall: {recall:.2f}%\n")
        
        # Matriz de Confusión
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benigno', 'Maligno'])
        
        # Dibujar en el sub-gráfico correspondiente
        disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
        axes[idx].set_title(nombre_escenario, fontsize=12)
        axes[idx].grid(False) # Quitar la cuadrícula de fondo de la matriz
        
    plt.tight_layout()
    plt.show()
    print("\n")

SyntaxError: f-string: expecting '}' (1483332934.py, line 2)